In [ ]:
import pickle
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from groq import Groq
import os
from dotenv import load_dotenv

load_dotenv(override=True)

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# Load saved data from Week 1
index = faiss.read_index("papers.index")
with open("papers_metadata.pkl", "rb") as f:
    data = pickle.load(f)

papers = data["papers"]
chunks = data["chunks"]

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Init Groq client
client = Groq(api_key=GROQ_API_KEY)

print("✅ Everything loaded!")
print(f"Papers in index: {index.ntotal}")
print(f"Groq key loaded: {'✅' if GROQ_API_KEY else '❌'}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Everything loaded!
Papers in index: 39
Groq key loaded: ✅


In [2]:
def retrieve(query, top_k=8):
    query_vec = model.encode([query]).astype("float32")
    distances, indices = index.search(query_vec, top_k)
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            "rank": i+1,
            "title": papers[idx]["title"],
            "year": papers[idx].get("year", "N/A"),
            "url": papers[idx].get("url", ""),
            "abstract": papers[idx]["abstract"]
        })
    return results

# Test it
results = retrieve("pre-strain effect on fracture toughness high-strength steel", top_k=8)
print(f"Retrieved {len(results)} papers\n")
for r in results:
    print(f"#{r['rank']} [{r['year']}] {r['title']}")

Retrieved 8 papers

#1 [2024] Influence of pre-strain on fracture toughness of 3rd generation advanced high strength steels
#2 [2020] Effects of pre-fatigue damage on mechanical properties of Q690 high-strength steel
#3 [2019] Fracture Assessment of Weld Joints of High-Strength Steel in Pre-Strained Condition
#4 [2025] Enhanced ductility and fracture classification maps for advanced high-strength steels considering local ductility and fracture toughness
#5 [] A novel approach to assess hydrogen embrittlement (HE) susceptibility and mechanisms in high strength martensitic steels
#6 [2020] Roles of Hydrogen Content and Pre-strain on Damage Evolution of TRIP-aided Bainitic Ferrite Steel
#7 [2024] Revealing the Role of Pre-Strain on the Microstructure and Mechanical Properties of a High-Mn Austenitic Steel
#8 [2022] Damage criterion approach to high‐strength steel RHS truss joints


In [13]:
def build_context(results):
    context = ""
    for r in results:
        context += f"\n---\nTitle: {r['title']} ({r['year']})\nURL: {r['url']}\nAbstract: {r['abstract']}\n"
    return context

def synthesize_field(topic, results):
    context = build_context(results)
    
    prompt = f"""You are a senior researcher with deep expertise in materials science and mechanical engineering.

Based on the following research papers, write a rigorous synthesis of the current state of the field on this topic:
TOPIC: {topic}

PAPERS:
{context}

Write a structured synthesis covering:
1. Key findings and established knowledge in the field
2. Main methodologies used (experimental and/or simulation)
3. How different findings connect and build on each other
4. Contradictions or inconsistencies across studies

Write at PhD/postdoc level. Be specific — cite paper titles when referencing findings.
Synthesis:"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.4,
        max_tokens=1500
    )
    return response.choices[0].message.content

topic = "pre-strain influence on toughness and failure behavior in high-strength steel"
results = retrieve(topic, top_k=8)
synthesis = synthesize_field(topic, results)
print(synthesis)

The current state of the field on the pre-strain influence on toughness and failure behavior in high-strength steel has been extensively investigated through various research studies. This synthesis aims to provide a comprehensive overview of the key findings, methodologies, and connections between different studies.

**Key Findings and Established Knowledge**

Research has shown that pre-strain can significantly influence the mechanical properties and failure behavior of high-strength steel. For instance, the study "Influence of pre-strain on fracture toughness of 3rd generation advanced high strength steels" found that pre-straining exerts a notable influence on the non-essential plastic work, but has little or no influence on the fracture properties of advanced high-strength steels (AHSS) [1]. In contrast, the study "Effects of pre-fatigue damage on mechanical properties of Q690 high-strength steel" revealed that pre-fatigue damage can lead to significant reductions in ultimate elon

In [7]:
# Cell 4: Research Gap Analysis
gap_analysis_query = "What are the open research questions and gaps in pre-strain effects on high-strength steel toughness?"

retrieved_for_gaps = retrieve(gap_analysis_query, top_k=8)

gap_prompt = f"""You are an expert research analyst. Based on the following papers about pre-strain influence on high-strength steel, identify:

1. **Underexplored areas**: What aspects have received little attention?
2. **Methodological gaps**: What experimental or simulation approaches are missing?
3. **Contradictions that need resolution**: Which conflicting findings need reconciliation?
4. **Practical/industry gaps**: What real-world problems remain unsolved?
5. **Emerging opportunities**: What new directions does recent work (2023–2025) point toward?

Papers:
{retrieved_for_gaps}

Be specific. Reference paper titles and years when relevant. Focus on gaps that a PhD researcher could realistically address."""

gap_response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": gap_prompt}],
    max_tokens=1500,
    temperature=0.4
)

print("=== RESEARCH GAPS ===\n")
print(gap_response.choices[0].message.content)

=== RESEARCH GAPS ===

Based on the provided papers, the following gaps and opportunities have been identified:

**1. Underexplored areas:**
- The effects of pre-strain on the microstructure and mechanical properties of high-strength steels under various environmental conditions, such as high temperatures or corrosive environments, have received little attention. For example, the study in "Revealing the Role of Pre-Strain on the Microstructure and Mechanical Properties of a High-Mn Austenitic Steel" (2024) focused on cryogenic impact toughness, but similar investigations at elevated temperatures are scarce.
- The influence of pre-strain on the weldability and welding properties of high-strength steels, particularly in the context of advanced welding techniques, is not well understood. The study in "Fracture Assessment of Weld Joints of High-Strength Steel in Pre-Strained Condition" (2019) highlights the importance of pre-strain on fracture resistance, but more research is needed on the

In [8]:
# Cell 5: Novel Hypothesis Generation
hypothesis_prompt = f"""You are a creative research scientist with deep expertise in materials science and structural engineering.

Based on this gap analysis of pre-strain effects on high-strength steels:

{gap_response.choices[0].message.content}

Generate 5 novel, specific, and testable research hypotheses that:
- Address one or more of the identified gaps
- Are realistic for a PhD research project (3-4 years)
- Combine ideas across gaps in unexpected ways
- Could lead to a publishable contribution

For each hypothesis, provide:
- **Hypothesis**: A clear, falsifiable statement
- **Gap addressed**: Which gap(s) from the analysis it targets
- **Why novel**: What makes this different from existing work
- **Proposed approach**: Brief experimental or simulation methodology
- **Expected impact**: Why this matters to the field

Be bold and creative. Think across disciplines if needed."""

hypothesis_response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": hypothesis_prompt}],
    max_tokens=2000,
    temperature=0.7
)

print("=== NOVEL RESEARCH HYPOTHESES ===\n")
print(hypothesis_response.choices[0].message.content)

=== NOVEL RESEARCH HYPOTHESES ===

Here are five novel research hypotheses that address the identified gaps in the study of pre-strain effects on high-strength steels:

**Hypothesis 1:**
**Hypothesis**: Pre-straining high-strength steels under environmentally controlled conditions (e.g., high temperatures, humidity) will lead to the formation of unique microstructural features that enhance their resistance to hydrogen embrittlement.
**Gap addressed**: Underexplored areas (1), Methodological gaps (2)
**Why novel**: This hypothesis combines the effects of pre-strain with environmental conditions to explore their synergistic impact on microstructure and mechanical properties, which has not been thoroughly investigated.
**Proposed approach**: Utilize in-situ microscopy and synchrotron X-ray diffraction to study the microstructural evolution of high-strength steels under pre-strain and various environmental conditions. Perform hydrogen embrittlement tests to evaluate the resistance of pre-s

In [16]:
# Cell 6: Save results to markdown report
from datetime import datetime

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
report_filename = f"research_report_{timestamp}.md"

# Handle both string and response object
gap_text = gap_response if isinstance(gap_response, str) else gap_response.choices[0].message.content
hypothesis_text = hypothesis_response if isinstance(hypothesis_response, str) else hypothesis_response.choices[0].message.content

report_content = f"""# AI Literature Research Report
**Topic:** Pre-strain influence on toughness and failure behavior in high-strength steel  
**Generated:** {datetime.now().strftime("%Y-%m-%d %H:%M")}  
**Papers analyzed:** {index.ntotal}

---

## 1. Field Synthesis

{synthesis}

---

## 2. Research Gaps

{gap_text}

---

## 3. Novel Research Hypotheses

{hypothesis_text}
"""

with open(report_filename, "w", encoding="utf-8") as f:
    f.write(report_content)

print(f"✅ Report saved to: {report_filename}")

✅ Report saved to: research_report_2026-03-26_22-43.md


In [21]:
# Cell 7: Add citations to report
import glob

papers_list = data['papers']

# Build a deduplicated references list
def get_top_papers(query, top_k=8):
    query_embedding = model.encode([query])
    distances, indices = index.search(query_embedding, top_k)
    refs = []
    seen_titles = set()
    for idx in indices[0]:
        if idx < len(papers_list):
            paper = papers_list[idx]
            title = paper.get('title', 'Unknown Title')
            if title not in seen_titles:
                seen_titles.add(title)
                refs.append({
                    "title": title,
                    "year": paper.get('year', 'n.d.') or 'n.d.',
                    "url": paper.get('url', ''),
                    "authors": paper.get('authors', '')
                })
    return refs

all_refs = {}
for query in [topic, gap_analysis_query]:
    for paper in get_top_papers(query, top_k=8):
        all_refs[paper['title']] = paper

# Format references section
references_md = "\n---\n\n## References\n\n"
for i, (title, paper) in enumerate(all_refs.items(), 1):
    url_part = f" — [{paper['url']}]({paper['url']})" if paper['url'] else ""
    references_md += f"{i}. **{title}** ({paper['year']}){url_part}\n"

# Append to latest report
report_files = sorted(glob.glob("research_report_*.md"), reverse=True)
latest_report = report_files[0]

with open(latest_report, "a", encoding="utf-8") as f:
    f.write(references_md)

print(f"✅ Citations added to: {latest_report}")
print(f"\n--- {len(all_refs)} references added ---")
for i, (title, paper) in enumerate(all_refs.items(), 1):
    print(f"{i}. {title} ({paper['year']})")

✅ Citations added to: research_report_2026-03-26_22-43.md

--- 9 references added ---
1. Influence of pre-strain on fracture toughness of 3rd generation advanced high strength steels (2024)
2. Effects of pre-fatigue damage on mechanical properties of Q690 high-strength steel (2020)
3. Enhanced ductility and fracture classification maps for advanced high-strength steels considering local ductility and fracture toughness (2025)
4. A novel approach to assess hydrogen embrittlement (HE) susceptibility and mechanisms in high strength martensitic steels (n.d.)
5. Roles of Hydrogen Content and Pre-strain on Damage Evolution of TRIP-aided Bainitic Ferrite Steel (2020)
6. Revealing the Role of Pre-Strain on the Microstructure and Mechanical Properties of a High-Mn Austenitic Steel (2024)
7. Fracture Assessment of Weld Joints of High-Strength Steel in Pre-Strained Condition (2019)
8. Damage criterion approach to high‐strength steel RHS truss joints (2022)
9. On the critical role of martensite ha

In [20]:
print("data keys:", list(data.keys()))
print("\nFirst value sample:")
first_key = list(data.keys())[0]
print(f"Key: {first_key}")
print(f"Type: {type(data[first_key])}")
print(f"Length: {len(data[first_key])}")
print(f"First item: {data[first_key][0]}")

data keys: ['papers', 'chunks']

First value sample:
Key: papers
Type: <class 'list'>
Length: 39
First item: {'title': 'A damage-mechanics model for fracture nucleation and propagation', 'abstract': 'In this paper a composite model for earthquake rupture initiation and propagation is proposed. The model includes aspects of damage mechanics, fiber-bundle models, and slider-block models. An array of elements is introduced in analogy to the fibers of a fiber bundle. Time to failure for each element is specified from a Poisson distribution. The hazard rate is assumed to have a power-law dependence on stress. When an element fails it is removed, the stress on a failed element is redistributed uniformly to a specified number of neighboring elements in a given range of interaction. Damage is defined to be the fraction of elements that have failed. Time to failure and modes of rupture propagation are determined as a function of the hazard-rate exponent and the range of interaction.', 'year': '